In [1]:
import torch
import torch.nn.functional as F
from flow_core import Pi0ActionExpert, compute_flow_matching_loss, sample_actions

# 配置参数
B, H, D_act, D_cond = 4, 16, 20, 128
model = Pi0ActionExpert(action_dim=D_act, horizon=H, cond_dim=D_cond).eval()

dummy_actions = torch.randn(B, H, D_act)
dummy_cond = torch.randn(B, D_cond)

def test_loss_implementation():
    # 我们通过固定随机种子来比对你的实现与标准逻辑
    torch.manual_seed(0)
    your_loss = compute_flow_matching_loss(model, dummy_actions, dummy_cond)
    
    # --- 内部标准参考 ---
    torch.manual_seed(0)
    eps = torch.randn_like(dummy_actions)
    
    torch.manual_seed(0) # 假设你在 sample_time 内部也用了这个 seed
    # 模拟 Beta(1.5, 1.0)
    dist = torch.distributions.Beta(torch.tensor(1.5), torch.tensor(1.0))
    t = dist.sample((B,)) * 0.999 + 0.001
    
    t_exp = t[:, None, None]
    target_x_t = t_exp * eps + (1 - t_exp) * dummy_actions
    target_u_t = eps - dummy_actions
    target_v_t = model(target_x_t, dummy_cond, t)
    expected_loss = F.mse_loss(target_u_t, target_v_t)
    # ------------------
    
    error = torch.abs(your_loss - expected_loss).item()
    print(f"Loss MSE Error: {error:.2e}")
    assert error < 1e-5, "Loss implementation deviates from pi0 standard!"
    print("✅ Loss implementation matches pi0 standard!")

test_loss_implementation()

Loss MSE Error: 0.00e+00
✅ Loss implementation matches pi0 standard!


In [2]:
def test_sampling_dimensions():
    torch.manual_seed(0)
    sampled_act = sample_actions(model, dummy_cond, D_act, H, num_steps=5)
    
    print(f"Sampled Action Shape: {sampled_act.shape}")
    assert sampled_act.shape == (B, H, D_act), "Incorrect output shape for Action Chunking!"
    
    # 简单检查值是否合理 (经过 1.0 -> 0.0 的积分，不应全是噪声)
    noise = torch.randn_like(sampled_act)
    dist_to_noise = F.mse_loss(sampled_act, noise).item()
    print(f"Distance to pure noise: {dist_to_noise:.4f}")
    assert dist_to_noise > 0.1, "Sampled actions look like pure noise. Check your Euler step!"
    print("✅ Sampling dimensions and basic logic look good!")

test_sampling_dimensions()

Sampled Action Shape: torch.Size([4, 16, 20])
Distance to pure noise: 2.1223
✅ Sampling dimensions and basic logic look good!
